# E1 by hand: quantiles, calibration counts and whole-day rounding

Four small checks, each tied to one claim in the report. Nothing here fits a model or adds a new analysis;
section 4 only recomputes a saved, registered comparison from the per-order files.

1. Enumerate the loss for a few delivery times: why the optimal promise is a quantile.
2. Rank residuals by hand: the difference between $k/n$ and $k/(n+1)$.
3. Round up, round down, subtract one day: which boundary conditions the formulas need.
4. Recompute one real comparison from the saved per-order files.

## How to run

- **Version:** E1 final repository version (main report v6). Sections 1–3 need only NumPy and pandas.
- **Where:** open this notebook from the `notebooks/` folder of the repository, so that `../outputs/e1/` resolves.
- **Inputs for section 4** (shipped in the repository; checked by SHA-256 below):

| File | SHA-256 |
|---|---|
| `outputs/e1/score/e_promises_per_order.csv.gz` | `0146a12a…5036a7` |
| `outputs/e1/final/c_predictions_per_order.csv.gz` | `f2aa85e8…b8b0ab` |
| `outputs/e1/freeze/FREEZE.json` | `3b62af1a…8328dd` |

If a file is missing or different, the next cell stops with a message naming it. Sections 1–3 still run.

In [1]:
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 120)

E1 = Path("../outputs/e1")
INPUTS = {
    "score/e_promises_per_order.csv.gz": "0146a12abd8028f6f0b88608539d87e7540c406d580d23f7262685daaa5036a7",
    "final/c_predictions_per_order.csv.gz": "f2aa85e82ec22dc604f8fa4271078070c11ec9640590ba4f706542fea4b8b0ab",
    "freeze/FREEZE.json": "3b62af1ac385c2b8c9299e2e757a7e57e9ab2079526c9152878a1f5bfe8328dd",
}
problems = []
for rel, expected in INPUTS.items():
    f = E1 / rel
    if not f.exists():
        problems.append(f"missing: {f.resolve()}")
    elif hashlib.sha256(f.read_bytes()).hexdigest() != expected:
        problems.append(f"different content: {f.resolve()}")
SECTION4_READY = not problems
print("section 4 inputs OK" if SECTION4_READY else
      "Section 4 will be skipped. Run from the repository's notebooks/ folder with the shipped outputs/e1/ files. "
      + "; ".join(problems))

section 4 inputs OK


## 1. Enumerating the loss

Ten orders with whole-day delivery times. With $c_\text{late} = 9$ and $c_\text{slack} = 1$, $q = 9/(9+1) = 0.9$.
For each promise $d$, average the loss over the ten orders and compare it with $F(d)$, the share delivered within $d$ days.

In [2]:
y = np.array([2, 3, 3, 4, 5, 5, 6, 8, 12, 20])
c_late, c_slack = 9.0, 1.0
q = c_late / (c_late + c_slack)

def avg_loss(d, y):
    return np.mean(c_late * np.maximum(y - d, 0) + c_slack * np.maximum(d - y, 0))

table = pd.DataFrame({"d": np.arange(1, 23)})
table["F(d)"] = [(y <= d).mean() for d in table.d]
table["avg loss R(d)"] = [avg_loss(d, y) for d in table.d]
table["R(d+1) - R(d)"] = table["avg loss R(d)"].shift(-1) - table["avg loss R(d)"]
table

,d,F(d),avg loss R(d),R(d+1) - R(d)
0,1,0.0,52.2,-9.0
1,2,0.1,43.2,-8.0
2,3,0.3,35.2,-6.0
3,4,0.4,29.2,-5.0
4,5,0.6,24.2,-3.0
5,6,0.7,21.2,-2.0
6,7,0.7,19.2,-2.0
7,8,0.8,17.2,-1.0
8,9,0.8,16.2,-1.0
9,10,0.8,15.2,-1.0


In [3]:
d_star = int(table.loc[table["F(d)"] >= q, "d"].min())          # smallest d with F(d) >= q
best = table.loc[np.isclose(table["avg loss R(d)"], table["avg loss R(d)"].min()), "d"].tolist()
print("smallest d with F(d) >= 0.9:", d_star)
print("all minimisers of R(d):     ", best)
print("mean delivery time:", y.mean(), "-> promise", int(np.ceil(y.mean())), "loss", avg_loss(np.ceil(y.mean()), y),
      "vs", avg_loss(d_star, y), "at d*")
assert d_star == best[0]

smallest d with F(d) >= 0.9: 12
all minimisers of R(d):      [12, 13, 14, 15, 16, 17, 18, 19, 20]
mean delivery time: 6.8 -> promise 7 loss 19.2 vs 13.2 at d*


**Reading.** $R(d+1)-R(d)$ is negative while $F(d) < 0.9$ and zero while $F(d) = 0.9$ exactly. Here $F(d)=0.9$
for $d = 12, \dots, 19$, so $R$ is flat from 12 to 20 and every promise from 12 to 20 days is optimal: the minimiser
is not unique when $F(d^\ast) = q$.
The smallest one, $d^\ast = 12$, is the lower 0.9-quantile, which is the rule the project uses. Promising the
rounded-up mean (7 days) costs more under this loss. That is a property of this loss and this sample, not a claim that
the mean can never coincide with the optimal quantile.

## 2. Ranking residuals by hand: $k/n$ versus $k/(n+1)$

A fixed predictor and nine calibration orders. With $q = 0.8$: $k = \lceil (n+1)q \rceil = \lceil 8 \rceil = 8$.

In [4]:
y_cal = np.array([7, 9, 6, 12, 8, 10, 15, 7, 11])
f_cal = np.array([8.0, 8.5, 6.2, 9.0, 8.1, 9.4, 10.0, 7.5, 9.9])     # predictions made before seeing these orders
q2 = 0.8
n = len(y_cal)
k = int(np.ceil((n + 1) * q2))
r = y_cal - f_cal
s = np.sort(r)[k - 1]                                                # k-th smallest residual
print("sorted residuals:", np.round(np.sort(r), 2))
print(f"n = {n}, k = {k}, shift s = r_(k) = {s:.2f}")
cover_cal = (y_cal <= f_cal + s).mean()
print(f"coverage on the calibration set = {cover_cal:.3f}  >=  k/n = {k/n:.3f}")
print(f"k/(n+1) = {k/(n+1):.3f}   (the guarantee for a NEW order, under exchangeability)")

sorted residuals: [-1.  -0.5 -0.2 -0.1  0.5  0.6  1.1  3.   5. ]
n = 9, k = 8, shift s = r_(k) = 3.00
coverage on the calibration set = 0.889  >=  k/n = 0.889
k/(n+1) = 0.800   (the guarantee for a NEW order, under exchangeability)


The calibration count $k/n$ is a fact about these nine orders. The number $k/(n+1)$ is a probability for a new
order, and it needs an assumption. A simulation makes the difference visible: draw exchangeable (here i.i.d.)
residuals many times, calibrate on nine, and check one new residual each time.

In [5]:
rng = np.random.default_rng(1)
reps = 200_000
R = rng.normal(size=(reps, n + 1))                     # n calibration residuals + 1 new one, i.i.d.
s_rep = np.sort(R[:, :n], axis=1)[:, k - 1]
cal_cov = (R[:, :n] <= s_rep[:, None]).mean(axis=1)
new_cov = (R[:, n] <= s_rep)
print(f"smallest calibration coverage over {reps:,} calibration sets: {cal_cov.min():.3f}  (k/n = {k/n:.3f})")
print(f"share of new orders covered: {new_cov.mean():.4f}  (k/(n+1) = {k/(n+1):.4f})")

smallest calibration coverage over 200,000 calibration sets: 0.889  (k/n = 0.889)
share of new orders covered: 0.8004  (k/(n+1) = 0.8000)


Every calibration set covers at least $k/n = 0.889$ of its own orders, but only $k/(n+1) = 0.800$ of new orders are
covered on average. The new order is the $(n+1)$-th exchangeable draw, so its rank among all $n+1$ residuals is uniform.

If the new residuals come from a different distribution, the calibration count does not change, but new-order
coverage does:

In [6]:
for label, shift_new in [("same distribution", 0.0), ("later deliveries slower (+1 sd)", 1.0), ("later deliveries faster (-1 sd)", -1.0)]:
    new = R[:, n] + shift_new
    print(f"{label:35s} calibration >= {cal_cov.min():.3f}   new-order coverage = {(new <= s_rep).mean():.3f}")

same distribution                   calibration >= 0.889   new-order coverage = 0.800
later deliveries slower (+1 sd)     calibration >= 0.889   new-order coverage = 0.473
later deliveries faster (-1 sd)     calibration >= 0.889   new-order coverage = 0.960


## 3. Round up, round down, subtract one day

For a continuous bound $U$ and a whole-number $Y$: $Y \le U \iff Y \le \lfloor U \rfloor$ (untruncated).
The project also applies a minimum of one day.

In [7]:
def round_up(u):   return np.maximum(1, np.ceil(u - 1e-9))     # registered rule
def round_down(u): return np.maximum(1, np.floor(u + 1e-9))
def minus_one(u):  return np.maximum(1, np.ceil(u - 1 - 1e-9))

cases = pd.DataFrame({
    "case": ["U on a whole number", "U = 1, Y = 1", "U < 1, Y = 1", "U just below Y", "U inside a day"],
    "U":    [10.0, 1.0, 0.4, 9.3, 9.6],
    "Y":    [10,   1,   1,   10,  9],
})
for name, fn in [("round up", round_up), ("round down", round_down), ("subtract one", minus_one)]:
    cases[name] = fn(cases.U.to_numpy()).astype(int)
cases["Y <= U"] = cases.Y <= cases.U
for name in ["round up", "round down", "subtract one"]:
    cases[f"covered ({name})"] = cases.Y <= cases[name]
cases

,case,U,Y,round up,round down,subtract one,Y <= U,covered (round up),covered (round down),covered (subtract one)
0,U on a whole number,10.0,10,10,10,9,True,True,True,False
1,"U = 1, Y = 1",1.0,1,1,1,1,True,True,True,True
2,"U < 1, Y = 1",0.4,1,1,1,1,False,True,True,True
3,U just below Y,9.3,10,10,9,9,False,True,False,False
4,U inside a day,9.6,9,10,9,9,True,True,True,True


- **U on a whole number:** subtracting one day promises 9 and misses an order that $U$ covers.
- **U = 1, Y = 1:** the one-day minimum rescues subtract-one ($\max\{1, 0\} = 1$), so it does not always under-cover.
- **U < 1, Y = 1:** $U$ itself does not cover the order, but every rule promises 1 day and covers it: the minimum can only enlarge coverage.
- **U just below Y (9.3 vs 10):** $U$ misses the order and rounding up covers it. That is the slack of rounding up: it
  adds less than one day to $U$, but that can change coverage. The one-day minimum is a separate adjustment.

The two counterexamples from the review, on a whole calibration set:

In [8]:
def calibrate(y, f, q=0.9):
    n = len(y); k = int(np.ceil((n + 1) * q)); return np.sort(y - f)[k - 1], k / n

for label, y_c, f_c in [("every order 10 days, forecast 0.5", np.full(100, 10.0), np.full(100, 0.5)),
                        ("every order 1 day,   forecast 0.5", np.full(100, 1.0),  np.full(100, 0.5))]:
    s_c, kn = calibrate(y_c, f_c)
    u = f_c + s_c
    print(f"{label}: U = {u[0]:.1f}, k/n = {kn:.2f} | covered: round up {np.mean(y_c <= round_up(u)):.0%}, "
          f"round down {np.mean(y_c <= round_down(u)):.0%}, subtract one {np.mean(y_c <= minus_one(u)):.0%}")

every order 10 days, forecast 0.5: U = 10.0, k/n = 0.91 | covered: round up 100%, round down 100%, subtract one 0%
every order 1 day,   forecast 0.5: U = 1.0, k/n = 0.91 | covered: round up 100%, round down 100%, subtract one 100%


## 4. One real comparison from the saved files

Registered H3: the shallow HGB policy $D^\ast$ against the matched state rule B1c on E. The saved report gives
$-0.32$ days and $+0.61$ coverage points. Recompute it from the per-order file, and rebuild $D^\ast$'s promise from
its prediction and its frozen shift.

In [9]:
import json
if not SECTION4_READY:
    raise SystemExit("Section 4 inputs are missing or different; see the check at the top.")
po = pd.read_csv(E1 / "score/e_promises_per_order.csv.gz")
cp = pd.read_csv(E1 / "final/c_predictions_per_order.csv.gz")
freeze = json.load(open(E1 / "freeze/FREEZE.json"))
y_e = po.actual_days.to_numpy()

s_D = freeze["shifts_by_policy_and_q"]["Dstar@0.90"]["shift_days"]
rebuilt = round_up(po["Dstar@0.90|pred"].to_numpy() + s_D)
print("rebuilt D* promise equals the saved one:", np.array_equal(rebuilt, po["Dstar@0.90|promise"].to_numpy()))

d_D, d_B = po["Dstar@0.90|promise"].to_numpy(), po["B1c@0.90|promise"].to_numpy()
print(f"E orders: {len(po):,}")
print(f"D* - B1c mean promised days: {d_D.mean() - d_B.mean():+.2f}")
print(f"D* - B1c coverage, points:   {((y_e <= d_D).mean() - (y_e <= d_B).mean()) * 100:+.2f}")

rebuilt D* promise equals the saved one: True
E orders: 19,950
D* - B1c mean promised days: -0.32
D* - B1c coverage, points:   +0.61


In [10]:
g = cp[cp.run == "Dstar@0.90"]
y_c, f_c = g.actual_days.to_numpy(), g.prediction_unrounded.to_numpy()
n_c = len(y_c); k_c = int(np.ceil((n_c + 1) * 0.9))
s_check = np.sort(y_c - f_c)[k_c - 1]
print(f"C: n = {n_c:,}, k = {k_c:,}, k/n = {k_c/n_c:.5f}, k/(n+1) = {k_c/(n_c+1):.5f}")
print(f"shift recomputed by hand = {s_check:.6f}, frozen shift = {s_D:.6f}")
print(f"D* coverage on C (rounded up) = {(y_c <= round_up(f_c + s_D)).mean():.4f}  >= k/n")
print(f"D* coverage on E (rounded up) = {(y_e <= d_D).mean():.4f}   <- an observed proportion on selected orders, not a guarantee")

C: n = 17,056, k = 15,352, k/n = 0.90009, k/(n+1) = 0.90004
shift recomputed by hand = 6.459148, frozen shift = 6.459148
D* coverage on C (rounded up) = 0.9114  >= k/n
D* coverage on E (rounded up) = 0.9710   <- an observed proportion on selected orders, not a guarantee


**Reading.** Recomputing the $k$-th residual by hand and matching the frozen shift is the direct implementation
check. Coverage on C at or above $k/n$ is only a necessary consequence of the rule: a shift wrongly set to 1,000 days
would pass it too. The calibration count on C is above $k/n$, as it must be. Coverage on E is far higher, which is an
empirical result for these selected orders. Whether C and E are exchangeable is exactly what this project cannot
assume.